# Phase 4: Cross-Encoder Fine-Tuning

**Architecture A** — Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` on 3 520 hard-case pairs (entity-alias oversampled ×3).

Three runs are defined below:
- **Run A** (Cell 3): baseline — 5 epochs, lr=2e-5, 3× oversampling → `cross_encoder_best`
- **Run B** (Cell 6): longer + lower LR — 10 epochs, lr=5e-6, same data → `cross_encoder_v2`
- **Run C** (Cell 7): longer + lower LR + heavier oversampling — 10 epochs, lr=5e-6, 5× → `cross_encoder_v3`

## Before you start
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. Run cells top-to-bottom. The runtime restarts after Cell 1 — that is expected.

---
## Cell 1 — Install dependencies

Installs `sentence-transformers` (the only extra dependency needed on Colab).
**After the install completes, the runtime will restart automatically.**
Then re-run from Cell 2 onwards — do **not** re-run Cell 1.

In [1]:
!pip install -q 'sentence-transformers>=3.0,<4.0' tqdm
print('Install complete. Runtime will restart — then continue from Cell 2.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.4 MB/s eta 0:00:00
Install complete. Runtime will restart — then continue from Cell 2.


---
## Cell 2 — Upload training data

**Option A (Google Drive — recommended):** Mount your Drive and point `DATA_DIR` to wherever you uploaded the `data/training/` folder.

**Option B (direct upload):** Use the file upload widget. Upload `cross_encoder_train.jsonl` and `cross_encoder_val.jsonl`.
They will land in `/content/` and the paths below are set accordingly.

Run the option that matches how you stored the files, then skip the other.

In [2]:
import json, pathlib

# ── Option A: Google Drive ────────────────────────────────────────────────
USE_DRIVE = True   # set False to use Option B (direct upload)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to match where you put the thesis repo on your Drive:
    DATA_DIR = pathlib.Path('/content/drive/MyDrive/thesis/data/training')
else:
    # ── Option B: Direct upload ───────────────────────────────────────────
    from google.colab import files
    print('Upload cross_encoder_train.jsonl and cross_encoder_val.jsonl')
    files.upload()
    DATA_DIR = pathlib.Path('/content')

TRAIN_PATH = DATA_DIR / 'cross_encoder_train.jsonl'
VAL_PATH   = DATA_DIR / 'cross_encoder_val.jsonl'

# Quick sanity check
train_rows = [json.loads(l) for l in TRAIN_PATH.read_text().splitlines()]
val_rows   = [json.loads(l) for l in VAL_PATH.read_text().splitlines()]
print(f'Train pairs : {len(train_rows)}')
print(f'Val pairs   : {len(val_rows)}')
print(f'Example     : {train_rows[0]}')

Mounted at /content/drive
Train pairs : 3520
Val pairs   : 540
Example     : {'query': 'Who was the director of City of Beautiful Nonsense?', 'passage': " This Beautiful City is a Canadian drama film directed by Ed Gass-Donnelly. It premiered at the Toronto International Film Festival in 2007 and had a general theatrical release in 2008. The film depicts the lives of five disparate characters in Downtown Toronto. Johnny (Aaron Poole) is a recovering crack cocaine addict trying to convince his prostitute girlfriend Pretty (Kristin Booth) to move with him to a new city so they can make a clean break from their old lives, while Harry (Noam Jenkins) and Carol (Caroline Cave) are a wealthy couple. Events are set in motion when Carol falls from the balcony of her condo in an apparent suicide attempt, landing just metres away from Johnny and Pretty in the alleyway below. She survives, but Peter (Stuart Hughes), a police detective, finds her and the group's lives begin to intertwine.", 'label'

---
## Cell 3 — Run A: Baseline training

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-12-v2` with:
- **5 epochs**, **lr=2e-5**, **3× entity_alias oversampling** (3 520 pairs)
- **Loss**: `BCEWithLogitsLoss` (compatible with 0.9 / 0.1 label-smoothed targets)
- **Best model selection**: saved when validation accuracy improves

Expected wall-clock time on a Colab T4: ~12–15 minutes.

In [3]:
import torch
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

# ── Run A hyperparameters ─────────────────────────────────────────────────
BASE_MODEL    = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
OUTPUT_PATH   = '/content/cross_encoder_best'
LEARNING_RATE = 2e-5
EPOCHS        = 5
BATCH_SIZE    = 16
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01
EVAL_STEPS    = 50

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'
print(f'Device: {device}  |  AMP: {use_amp}')

model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=512)

train_examples = [
    InputExample(texts=[r['query'], r['passage']], label=float(r['label']))
    for r in train_rows
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

val_examples = [
    InputExample(texts=[r['query'], r['passage']], label=1 if r['label'] > 0.5 else 0)
    for r in val_rows
]
evaluator = CEBinaryClassificationEvaluator.from_input_examples(val_examples, name='val')

total_steps  = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
print(f'Total steps: {total_steps}  |  Warmup: {warmup_steps}')

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    loss_fct=torch.nn.BCEWithLogitsLoss(),
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LEARNING_RATE},
    weight_decay=WEIGHT_DECAY,
    evaluation_steps=EVAL_STEPS,
    output_path=OUTPUT_PATH,
    save_best_model=True,
    use_amp=use_amp,
    show_progress_bar=True,
)
print(f'\nRun A complete. Best model at {OUTPUT_PATH}')

Device: cuda  |  AMP: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]


Run A complete. Best model at /content/cross_encoder_best


---
## Cell 4 — Evaluate on validation set

Loads the best saved model and computes binary classification accuracy on the val set.
This is a quick sanity check before downloading — the full test evaluation runs locally
via `scripts/eval_cross_encoder.py` on the 136 held-out test cases.

In [4]:
import math

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

best_model = CrossEncoder(OUTPUT_PATH)

val_pairs   = [[r['query'], r['passage']] for r in val_rows]
val_labels  = [1 if r['label'] > 0.5 else 0 for r in val_rows]
raw_scores  = best_model.predict(val_pairs, show_progress_bar=True)
pred_labels = [1 if sigmoid(float(s)) >= 0.5 else 0 for s in raw_scores]

accuracy  = sum(p == g for p, g in zip(pred_labels, val_labels)) / len(val_labels)
tp = sum(p == 1 and g == 1 for p, g in zip(pred_labels, val_labels))
fp = sum(p == 1 and g == 0 for p, g in zip(pred_labels, val_labels))
fn = sum(p == 0 and g == 1 for p, g in zip(pred_labels, val_labels))
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f'Val accuracy  : {accuracy:.4f}')
print(f'Val precision : {precision:.4f}')
print(f'Val recall    : {recall:.4f}')
print(f'Val F1        : {f1:.4f}')
print(f'Val pairs     : {len(val_labels)}')

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Val accuracy  : 0.8907
Val precision : 0.7209
Val recall    : 0.9185
Val F1        : 0.8078
Val pairs     : 540


---
## Cell 5 — Save Run A model to Google Drive

Copies the fine-tuned model from Colab's ephemeral `/content/` to your Google Drive
so it survives runtime disconnects.

After downloading, place the folder at `models/cross_encoder_best/` in the thesis repo
root, then run:
```bash
python scripts/eval_cross_encoder.py
```

In [5]:
import shutil, pathlib

# Mount Drive if not already mounted
try:
    pathlib.Path('/content/drive/MyDrive').stat()
except FileNotFoundError:
    from google.colab import drive
    drive.mount('/content/drive')

# Adjust destination to match your Drive layout
DRIVE_DEST = pathlib.Path('/content/drive/MyDrive/thesis/models/cross_encoder_best')
DRIVE_DEST.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_DEST.exists():
    shutil.rmtree(DRIVE_DEST)  # overwrite previous run
shutil.copytree(OUTPUT_PATH, str(DRIVE_DEST))

print(f'Run A saved to Google Drive: {DRIVE_DEST}')
print('Files:', list(DRIVE_DEST.iterdir()))

Run A saved to Google Drive: /content/drive/MyDrive/thesis/models/cross_encoder_best
Files: [PosixPath('/content/drive/MyDrive/thesis/models/cross_encoder_best/CEBinaryClassificationEvaluator_val_results.csv'), PosixPath('/content/drive/MyDrive/thesis/models/cross_encoder_best/vocab.txt'), PosixPath('/content/drive/MyDrive/thesis/models/cross_encoder_best/model.safetensors'), PosixPath('/content/drive/MyDrive/thesis/models/cross_encoder_best/config.json'), PosixPath('/content/drive/MyDrive/thesis/models/cross_encoder_best/special_tokens_map.json'), PosixPath('/content/drive/MyDrive/thesis/models/cross_encoder_best/tokenizer_config.json'), PosixPath('/content/drive/MyDrive/thesis/models/cross_encoder_best/tokenizer.json')]


---
## Cell 6 — Run B: Longer training, lower LR (same data)

Hypothesis: Run A may not have converged — 10 epochs at a lower LR (5e-6) lets the
model refine its decision boundary without overstepping. Same 3 520 training pairs.

Changes vs Run A:
- **epochs**: 5 → **10**
- **learning_rate**: 2e-5 → **5e-6**
- Output saved to `cross_encoder_v2`

Expected wall-clock time on a Colab T4: ~25–30 minutes.

> **Requires Cell 2 to have been run** (uses `train_rows` / `val_rows`).

In [6]:
import torch, shutil, pathlib
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

# ── Run B hyperparameters ─────────────────────────────────────────────────
BASE_MODEL      = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
OUTPUT_PATH_B   = '/content/cross_encoder_v2'
LEARNING_RATE_B = 5e-6   # finer convergence
EPOCHS_B        = 10
BATCH_SIZE      = 16
WARMUP_RATIO    = 0.1
WEIGHT_DECAY    = 0.01
EVAL_STEPS      = 50

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'
print(f'Run B  |  Device: {device}  |  LR: {LEARNING_RATE_B}  |  Epochs: {EPOCHS_B}')
print(f'Train pairs: {len(train_rows)}  |  Val pairs: {len(val_rows)}')

model_b = CrossEncoder(BASE_MODEL, num_labels=1, max_length=512)

train_examples_b = [
    InputExample(texts=[r['query'], r['passage']], label=float(r['label']))
    for r in train_rows
]
train_dl_b = DataLoader(train_examples_b, shuffle=True, batch_size=BATCH_SIZE)

val_examples_b = [
    InputExample(texts=[r['query'], r['passage']], label=1 if r['label'] > 0.5 else 0)
    for r in val_rows
]
evaluator_b = CEBinaryClassificationEvaluator.from_input_examples(val_examples_b, name='val')

total_steps_b  = len(train_dl_b) * EPOCHS_B
warmup_steps_b = int(total_steps_b * WARMUP_RATIO)
print(f'Total steps: {total_steps_b}  |  Warmup: {warmup_steps_b}')

model_b.fit(
    train_dataloader=train_dl_b,
    evaluator=evaluator_b,
    epochs=EPOCHS_B,
    loss_fct=torch.nn.BCEWithLogitsLoss(),
    warmup_steps=warmup_steps_b,
    optimizer_params={'lr': LEARNING_RATE_B},
    weight_decay=WEIGHT_DECAY,
    evaluation_steps=EVAL_STEPS,
    output_path=OUTPUT_PATH_B,
    save_best_model=True,
    use_amp=use_amp,
    show_progress_bar=True,
)
print(f'\nRun B complete. Best model at {OUTPUT_PATH_B}')

# Save to Drive
drive_dest_b = pathlib.Path('/content/drive/MyDrive/thesis/models/cross_encoder_v2')
drive_dest_b.parent.mkdir(parents=True, exist_ok=True)
if drive_dest_b.exists():
    shutil.rmtree(drive_dest_b)
shutil.copytree(OUTPUT_PATH_B, str(drive_dest_b))
print(f'Run B saved to Google Drive: {drive_dest_b}')

Run B  |  Device: cuda  |  LR: 5e-06  |  Epochs: 10
Train pairs: 3520  |  Val pairs: 540
Total steps: 2200  |  Warmup: 220


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]

Iteration:   0%|          | 0/220 [00:00<?, ?it/s]


Run B complete. Best model at /content/cross_encoder_v2
Run B saved to Google Drive: /content/drive/MyDrive/thesis/models/cross_encoder_v2


In [8]:
import json, math
from sentence_transformers import CrossEncoder

model_b = CrossEncoder('/content/cross_encoder_v2')
val_rows = [json.loads(l) for l in open('/content/drive/MyDrive/thesis/data/training/cross_encoder_val.jsonl')]

correct = 0
for r in val_rows:
    score = model_b.predict([(r['query'], r['passage'])])[0]
    pred = 1 if 1/(1+math.exp(-score)) >= 0.5 else 0
    gold = 1 if r['label'] > 0.5 else 0
    if pred == gold:
        correct += 1

print(f"Run B val accuracy: {correct/len(val_rows):.4f}")

Run B val accuracy: 0.8685


---
## Cell 7 — Run C: Longer training, lower LR, 5× entity_alias oversampling

Hypothesis: entity_alias detection stalls at 65% because the model sees too few
examples relative to other trap types. Pushing oversampling from 3× to **5×**
(4 960 total pairs) forces the model to spend more gradient steps on hard cases.

Changes vs Run A:
- **epochs**: 5 → **10**
- **learning_rate**: 2e-5 → **5e-6**
- **entity_alias oversampling**: 3× → **5×** (data regenerated inline)
- Output saved to `cross_encoder_v3`

Expected wall-clock time on a Colab T4: ~30–35 minutes.

> **Requires the original hard-case splits on Drive** at
> `thesis/data/hard_cases/splits/`. No need to re-upload anything else.

In [9]:
import json, random, torch, shutil, pathlib
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

# ── Regenerate training data with 5× entity_alias oversampling ────────────
OVERSAMPLE_5X = 5
LABEL_POS, LABEL_NEG = 0.9, 0.1

HARD_CASES_DIR = pathlib.Path('/content/drive/MyDrive/thesis/data/hard_cases/splits')
train_raw = [json.loads(l) for l in (HARD_CASES_DIR / 'train.jsonl').read_text().splitlines()]
val_raw   = [json.loads(l) for l in (HARD_CASES_DIR / 'val.jsonl').read_text().splitlines()]

def make_pairs(cases, oversample=True, multiplier=OVERSAMPLE_5X):
    pairs = []
    for c in cases:
        q, tt = c['question'], c.get('trap_type', 'unknown')
        base = [
            {'query': q, 'passage': c['gold_passage'],           'label': LABEL_POS},
            {'query': q, 'passage': c['trap_passage'],           'label': LABEL_NEG},
            {'query': q, 'passage': c['irrelevant_passages'][0], 'label': LABEL_NEG},
            {'query': q, 'passage': c['irrelevant_passages'][1], 'label': LABEL_NEG},
        ]
        m = multiplier if (oversample and tt == 'entity_alias') else 1
        pairs.extend(base * m)
    return pairs

random.seed(42)
train_rows_c = make_pairs(train_raw, oversample=True)
random.shuffle(train_rows_c)
val_rows_c   = make_pairs(val_raw,   oversample=False)
print(f'Train pairs (5×): {len(train_rows_c)}  |  Val pairs: {len(val_rows_c)}')

# ── Run C hyperparameters ─────────────────────────────────────────────────
BASE_MODEL      = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
OUTPUT_PATH_C   = '/content/cross_encoder_v3'
LEARNING_RATE_C = 5e-6
EPOCHS_C        = 10
BATCH_SIZE      = 16
WARMUP_RATIO    = 0.1
WEIGHT_DECAY    = 0.01
EVAL_STEPS      = 50

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'
print(f'Run C  |  Device: {device}  |  LR: {LEARNING_RATE_C}  |  Epochs: {EPOCHS_C}  |  Oversample: {OVERSAMPLE_5X}×')

model_c = CrossEncoder(BASE_MODEL, num_labels=1, max_length=512)

train_examples_c = [
    InputExample(texts=[r['query'], r['passage']], label=float(r['label']))
    for r in train_rows_c
]
train_dl_c = DataLoader(train_examples_c, shuffle=True, batch_size=BATCH_SIZE)

val_examples_c = [
    InputExample(texts=[r['query'], r['passage']], label=1 if r['label'] > 0.5 else 0)
    for r in val_rows_c
]
evaluator_c = CEBinaryClassificationEvaluator.from_input_examples(val_examples_c, name='val')

total_steps_c  = len(train_dl_c) * EPOCHS_C
warmup_steps_c = int(total_steps_c * WARMUP_RATIO)
print(f'Total steps: {total_steps_c}  |  Warmup: {warmup_steps_c}')

model_c.fit(
    train_dataloader=train_dl_c,
    evaluator=evaluator_c,
    epochs=EPOCHS_C,
    loss_fct=torch.nn.BCEWithLogitsLoss(),
    warmup_steps=warmup_steps_c,
    optimizer_params={'lr': LEARNING_RATE_C},
    weight_decay=WEIGHT_DECAY,
    evaluation_steps=EVAL_STEPS,
    output_path=OUTPUT_PATH_C,
    save_best_model=True,
    use_amp=use_amp,
    show_progress_bar=True,
)
print(f'\nRun C complete. Best model at {OUTPUT_PATH_C}')

# Save to Drive
drive_dest_c = pathlib.Path('/content/drive/MyDrive/thesis/models/cross_encoder_v3')
drive_dest_c.parent.mkdir(parents=True, exist_ok=True)
if drive_dest_c.exists():
    shutil.rmtree(drive_dest_c)
shutil.copytree(OUTPUT_PATH_C, str(drive_dest_c))
print(f'Run C saved to Google Drive: {drive_dest_c}')

Train pairs (5×): 4512  |  Val pairs: 540
Run C  |  Device: cuda  |  LR: 5e-06  |  Epochs: 10  |  Oversample: 5×
Total steps: 2820  |  Warmup: 282


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Iteration:   0%|          | 0/282 [00:00<?, ?it/s]

Iteration:   0%|          | 0/282 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/282 [00:00<?, ?it/s]

Iteration:   0%|          | 0/282 [00:00<?, ?it/s]

Iteration:   0%|          | 0/282 [00:00<?, ?it/s]

Iteration:   0%|          | 0/282 [00:00<?, ?it/s]


Run C complete. Best model at /content/cross_encoder_v3
Run C saved to Google Drive: /content/drive/MyDrive/thesis/models/cross_encoder_v3


In [10]:
model_c = CrossEncoder('/content/cross_encoder_v3')

correct = 0
for r in val_rows:
    score = model_c.predict([(r['query'], r['passage'])])[0]
    pred = 1 if 1/(1+math.exp(-score)) >= 0.5 else 0
    gold = 1 if r['label'] > 0.5 else 0
    if pred == gold:
        correct += 1

print(f"Run C val accuracy: {correct/len(val_rows):.4f}")

Run C val accuracy: 0.8685


---
## Cell 8 — Run D: v2 data (Run A config — 6 832 pairs)

Trains on the **expanded v2 dataset** (PopQA 903 + EntityQuestions 511 = 1 414 cases,
merged and re-split). Training config is identical to the Phase 4 winner (Run A):
- **5 epochs**, **lr=2e-5**, **3× entity_alias oversampling**
- Training pairs: **6 832** (vs 3 520 for Run A)

Key change: entity_alias cases in training jump from 124 → 359 (×2.9), directly
targeting the structural 65.4% detection ceiling seen in Phase 4.

Upload `cross_encoder_train_v2.jsonl` and `cross_encoder_val_v2.jsonl` to Drive at:
`/content/drive/MyDrive/thesis/data/training/`

Expected wall-clock time on a Colab T4: ~25–30 minutes.

> **Requires Cell 1 (install) to have been run and the runtime restarted.**

In [ ]:
import json, pathlib, random, shutil, torch
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

# ── Load v2 training data from Drive ─────────────────────────────────────
try:
    pathlib.Path('/content/drive/MyDrive').stat()
except FileNotFoundError:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR_V2 = pathlib.Path('/content/drive/MyDrive/thesis/data/training')
TRAIN_PATH_V2 = DATA_DIR_V2 / 'cross_encoder_train_v2.jsonl'
VAL_PATH_V2   = DATA_DIR_V2 / 'cross_encoder_val_v2.jsonl'

train_rows_d = [json.loads(l) for l in TRAIN_PATH_V2.read_text().splitlines()]
val_rows_d   = [json.loads(l) for l in VAL_PATH_V2.read_text().splitlines()]
print(f'Train pairs (v2): {len(train_rows_d)}')
print(f'Val   pairs (v2): {len(val_rows_d)}')

# ── Run D hyperparameters — identical to Run A (Phase 4 winner) ───────────
BASE_MODEL    = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
OUTPUT_PATH_D = '/content/cross_encoder_best_v2'
LEARNING_RATE = 2e-5   # same as Run A
EPOCHS        = 5      # same as Run A
BATCH_SIZE    = 16
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01
EVAL_STEPS    = 50

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'
print(f'Device: {device}  |  LR: {LEARNING_RATE}  |  Epochs: {EPOCHS}')

model_d = CrossEncoder(BASE_MODEL, num_labels=1, max_length=512)

train_examples_d = [
    InputExample(texts=[r['query'], r['passage']], label=float(r['label']))
    for r in train_rows_d
]
train_dl_d = DataLoader(train_examples_d, shuffle=True, batch_size=BATCH_SIZE)

val_examples_d = [
    InputExample(texts=[r['query'], r['passage']], label=1 if r['label'] > 0.5 else 0)
    for r in val_rows_d
]
evaluator_d = CEBinaryClassificationEvaluator.from_input_examples(val_examples_d, name='val')

total_steps_d  = len(train_dl_d) * EPOCHS
warmup_steps_d = int(total_steps_d * WARMUP_RATIO)
print(f'Total steps: {total_steps_d}  |  Warmup: {warmup_steps_d}')

model_d.fit(
    train_dataloader=train_dl_d,
    evaluator=evaluator_d,
    epochs=EPOCHS,
    loss_fct=torch.nn.BCEWithLogitsLoss(),
    warmup_steps=warmup_steps_d,
    optimizer_params={'lr': LEARNING_RATE},
    weight_decay=WEIGHT_DECAY,
    evaluation_steps=EVAL_STEPS,
    output_path=OUTPUT_PATH_D,
    save_best_model=True,
    use_amp=use_amp,
    show_progress_bar=True,
)
print(f'\nRun D complete. Best model at {OUTPUT_PATH_D}')

# Evaluate
import math

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

best_d = CrossEncoder(OUTPUT_PATH_D)
val_pairs_d  = [[r['query'], r['passage']] for r in val_rows_d]
val_labels_d = [1 if r['label'] > 0.5 else 0 for r in val_rows_d]
raw_d = best_d.predict(val_pairs_d, show_progress_bar=True)
pred_d = [1 if sigmoid(float(s)) >= 0.5 else 0 for s in raw_d]
acc_d = sum(p == g for p, g in zip(pred_d, val_labels_d)) / len(val_labels_d)
print(f'Run D val accuracy: {acc_d:.4f}  (Run A reference: 0.8907)')

# ── Save to Drive ─────────────────────────────────────────────────────────
DRIVE_DEST_D = pathlib.Path('/content/drive/MyDrive/thesis/models/cross_encoder_v2')
DRIVE_DEST_D.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_DEST_D.exists():
    shutil.rmtree(DRIVE_DEST_D)
shutil.copytree(OUTPUT_PATH_D, str(DRIVE_DEST_D))
print(f'Run D saved to Drive: {DRIVE_DEST_D}')